In [ ]:
# Step 1: Install Required Libraries

# Step 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

#  Step 3: Copy Model to Colab  OR USE DIRECT

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
%%writefile app.py
from fastapi import FastAPI, Request
from pydantic import BaseModel # its decide - which type of input must be take 
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch
import re
from fastapi.templating import Jinja2Templates # UI
from fastapi.responses import HTMLResponse
from fastapi.staticfiles import StaticFiles

# initialize fastapi app
app = FastAPI(title="Text Summarizer App", description="Text Summarization using T5", version="1.0")

# model & tokenizer
model = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/Projects/text_summarizer/saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("/content/drive/MyDrive/Projects/text_summarizer/saved_summary_model")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

templates = Jinja2Templates(directory=".")

# Input schema for dialogue => string
class DialogueInput(BaseModel):
    dialogue: str

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags
    test = text.strip().lower()
    return text

def summarize_dialogue(dialogue: str) -> str:
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt" # it returns - Pytorch tensors as inputs
        # by default Hugging face model is Pytorch model
    ).to(device)

    # generate the summary => token ids (generate token ids, not actual summary text)
    model.to(device) # ensure that our data and model on same device
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4, # transformer generate 4 seq of out(summary) -> and choose one best
        early_stopping=True # after get EOS -> stop: and choose one best
    )
    
    # token ids convert to summary => decoding
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # 
    return summary

# API Endpoints
@app.post("/summarize/") # client send -> server | endpoint - '/summarize/'
async def summarize(dialogue_input: DialogueInput):
    summary = summarize_dialogue(dialogue_input.dialogue)
    return {"summary": summary} # return to client

@app.get("/", response_class=HTMLResponse)  # client get <- server | endpoint - '/'
async def home(request: Request):
    return templates.TemplateResponse("index.html", {"request": request})

Writing app.py


In [7]:
# 5. Add HTML UI

In [12]:
# Step 6: Verify
!ls

app.ipynb  app.py  drive  index.html  __pycache__  sample_data


In [13]:
# Step 7: Start FastAPI Server
process = subprocess.Popen(["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"])

In [14]:
# Step 8: Start Ngrok Tunnel

ngrok.set_auth_token("3BlOdCZLPdoL5aVUyJiSYuxeHmg_4vvVc6aE8bhLW7GGqeK96")

public_url = ngrok.connect(8000)
print("Public URL: ", public_url)

Public URL:  NgrokTunnel: "https://nonparochial-hyperdolichocephalic-denver.ngrok-free.dev" -> "http://localhost:8000"
